# Session 4: files and the csv module

Until now every value has been typed into the code. Real data lives on disk.

## First: where am I?

Notebooks and scripts both look for files **relative to the folder they are
running in**, and that folder is not always the one you expect.

This is the first thing to print when a file will not open. The file is
nearly always fine; Python is just looking somewhere else.

In [ ]:
import os

print(os.getcwd())

For this course, run everything from the **repository root**, which is where
the path below is counted from. If the cell after this one fails with a
`FileNotFoundError`, that is what has gone wrong.

In VS Code, a notebook runs in the folder of the notebook by default, so we
nudge it up to the repository root first. `os.chdir` changes directory, and
the `if` makes the cell safe to run twice.

In [ ]:
from pathlib import Path

# Walk up until we find the folder with the data/ directory in it.
here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent

os.chdir(here)
print("working from:", Path.cwd())

## Opening a file

Two rules, both worth following every single time.

In [ ]:
PLAYS = "data/clean/plays.csv"

with open(PLAYS, encoding="utf-8") as f:
    first_line = f.readline()

print(first_line)

**`with`** closes the file for you, even if your code fails halfway through.
An unclosed file can hold a lock on Windows and lose data you thought you
had written.

**`encoding="utf-8"`** tells Python how the text is stored. Without it,
Python guesses based on your operating system: usually right on a Mac,
usually wrong on Windows, and the symptom is a `UnicodeDecodeError` the first
time somebody's name has an accent in it. Our data contains
`Salon Mécanique`, so this matters today.

## Reading a CSV into dictionaries

In [ ]:
import csv

with open(PLAYS, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(len(rows))
print(rows[0])

`DictReader` reads the header row and uses it for the keys. What comes back
is exactly the shape from the last notebook: **a list of dictionaries, one
per row**. You already know how to loop over that.

In [ ]:
print(list(rows[0].keys()))
print(rows[0]["genre"])

Peek at a few rows rather than printing two thousand lines.

In [ ]:
for row in rows[:3]:
    print(row["played_at"], row["artist_name"], row["minutes_played"])

**Why `list(...)` around `DictReader`?** Without it you get a reader you can
only walk through once, and a second loop over it silently does nothing at
all. That is a genuinely nasty bug, because nothing complains.

## Everything from a CSV arrives as text

In [ ]:
print(rows[0]["minutes_played"])
print(type(rows[0]["minutes_played"]))

It looks like a number and it is text. Adding it up fails. Uncomment to see
the error, which is the same lesson as `input()` in session 1.

In [ ]:
# total = 0
# for row in rows:
#     total += row["minutes_played"]
# TypeError: unsupported operand type(s) for +=: 'int' and 'str'

In [ ]:
total_minutes = 0.0
for row in rows:
    total_minutes += float(row["minutes_played"])     # convert, then add

print(f"{total_minutes:,.0f} minutes")

A file has no idea what a number is. It contains characters. `4.65` in a CSV
is the text `"4.65"` until you say `float()`.

This is the most common source of bugs when people start working with files.
When something numeric behaves strangely, print its `type()` first.

## The pattern of the day: counting into a dictionary

In [ ]:
counts = {}                                    # empty to start

for row in rows:
    genre = row["genre"]
    counts[genre] = counts.get(genre, 0) + 1

for genre, n in sorted(counts.items(), key=lambda pair: pair[1], reverse=True):
    print(f"{genre:12} {n:>5}")

Look at what that is: **the accumulate pattern from session 2**, with a
dictionary instead of a single total.

In session 6 you will write the same question as `GROUP BY genre`, and in
session 8 as `groupby("genre")`. Same idea, three notations.

## Writing a CSV back out

In [ ]:
os.makedirs("output", exist_ok=True)      # "w" fails if the folder is missing

# A list comprehension: build a new list from an old one, keeping matches.
# The same thing as a for loop with an append, on one line.
jazz = [row for row in rows if row["genre"] == "Jazz"]
print(len(jazz))

with open("output/jazz.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(jazz)

print("wrote output/jazz.csv")

Three arguments on that `open` worth remembering:

* `"w"` means write, and it **replaces** the file without asking
* `newline=""` stops blank lines appearing between rows on Windows
* `fieldnames` sets the column order

## Meet the listening log

This is the dataset for the rest of the course.

| Column | Type | Example |
|---|---|---|
| `play_id` | number | 1 |
| `played_at` | date | 2024-01-01 |
| `artist_name` | text | Fjord & Flint |
| `track_name` | text | Golden Garden |
| `genre` | category | Folk |
| `country` | category | Norway |
| `minutes_played` | number | 4.65 |
| `device` | category | phone |
| `skipped` | 0 or 1 | 0 |

2,183 plays, 40 artists, 8 genres, 5 devices, from 1 January 2024 to
31 December 2025. Small enough to check by hand, big enough that you would
not want to.

You will meet it four more times: as a SQLite database in sessions 6 and 7,
as a pandas DataFrame in session 8, as a deliberately messy file in session
9, and as the input to a full pipeline in session 10.

It is generated by a script, so if you ever break it:

```
python3 data/scripts/build_dataset.py
```

puts it back exactly as it was.

In [ ]:
import collections

print("rows:    ", len(rows))
print("artists: ", len(set(row["artist_name"] for row in rows)))
print("dates:   ", min(r["played_at"] for r in rows), "to",
      max(r["played_at"] for r in rows))
print("devices: ", collections.Counter(r["device"] for r in rows).most_common())

That last line uses `collections.Counter`, which does the counting pattern
for you in one call. It exists, it is good, and it was worth writing the
dictionary version by hand first so you know what it is doing.

## Next

`03-explore-plays-exercise.ipynb`: three questions about this data, answered
by you.